In [ ]:
#IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Amazon-themed plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(['#FF9900', '#232F3E', '#146EB4'])

print('✅ Libraries loaded successfully!')

## Step 1: Load the Dataset

In [ ]:
# Load dataset
df = pd.read_csv('AmazonSalesData.csv')

print(f'✅ Dataset loaded successfully')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\n📋 First 5 rows:')
df.head()

## Step 2: Examine Data Types & Structure

In [ ]:
# Check column data types and non-null counts
print('📊 Dataset Info:')
df.info()

print('\n📋 Column List:')
print(df.columns.tolist())

## Step 3: Check for Missing Values

In [ ]:
#MISSING VALUES CHECK 
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if len(missing_df) > 0:
    print('⚠️ Columns with missing values:')
    print(missing_df)
else:
    print('✅ No missing values found in any column!')
    print('   Dataset is clean and complete.')

## Step 4: Check for Duplicates

In [ ]:
#DUPLICATE CHECK
duplicates = df.duplicated().sum()
duplicate_orders = df['Order ID'].duplicated().sum()

print(f'Full row duplicates: {duplicates}')
print(f'Duplicate Order IDs: {duplicate_orders}')

if duplicates == 0 and duplicate_orders == 0:
    print('\n✅ No duplicates found — data integrity confirmed!')

## Step 5: Convert Date Columns

In [ ]:
# PARSE DATE COLUMNS 
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')

print('✅ Date columns converted to datetime')
print(f'   Order Date range: {df["Order Date"].min()} → {df["Order Date"].max()}')
print(f'   Ship Date range:  {df["Ship Date"].min()} → {df["Ship Date"].max()}')

## Step 6: Decode Order Priority Codes

In [ ]:
    #DECODE ORDER PRIORITY 
    # Original codes: H, M, L, C — replaced with readable labels
    priority_map = {'H': 'High', 'M': 'Medium', 'L': 'Low', 'C': 'Critical'}
    df['Order Priority'] = df['Order Priority'].map(priority_map)

    print('✅ Order Priority codes decoded:')
    print(df['Order Priority'].value_counts())

## Step 7: Create Calculated Columns for Time Series Analysis

In [ ]:
# DERIVE TIME-BASED COLUMNS 
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.month
df['Month Name'] = df['Order Date'].dt.strftime('%b')
df['Quarter']    = df['Order Date'].dt.quarter
df['Day Name']   = df['Order Date'].dt.day_name()

#DERIVE FULFILMENT TIME
df['Days to Ship'] = (df['Ship Date'] - df['Order Date']).dt.days

#DERIVE PROFIT MARGIN
df['Profit Margin %'] = (df['Total Profit'] / df['Total Revenue'] * 100).round(2)

print('✅ New calculated columns added:')
print('   - Year, Month, Quarter, Month Name, Day Name')
print('   - Days to Ship (Ship Date − Order Date)')
print('   - Profit Margin %')

## Step 8: Descriptive Statistics — Key KPIs

In [ ]:
#KEY METRICS SUMMARY
print('💰 KEY BUSINESS METRICS:\n')
print(f'   Total Revenue:      ${df["Total Revenue"].sum():,.2f}')
print(f'   Total Profit:       ${df["Total Profit"].sum():,.2f}')
print(f'   Total Cost:         ${df["Total Cost"].sum():,.2f}')
print(f'   Total Units Sold:   {df["Units Sold"].sum():,}')
print(f'   Avg Profit Margin:  {df["Profit Margin %"].mean():.2f}%')
print(f'   Total Orders:       {len(df):,}')
print(f'   Avg Order Value:    ${df["Total Revenue"].mean():,.2f}')
print(f'   Avg Days to Ship:   {df["Days to Ship"].mean():.1f} days')

print('\n📊 Numerical Columns Summary:')
df[['Total Revenue', 'Total Profit', 'Units Sold', 'Profit Margin %', 'Days to Ship']].describe().round(2)

## Step 9: Categorical Variables Overview

In [ ]:
#CATEGORICAL ANALYSIS 
cat_cols = ['Region', 'Country', 'Item Type', 'Sales Channel', 'Order Priority']

for col in cat_cols:
    print(f'\n🔹 {col}: {df[col].nunique()} unique values')
    print(df[col].value_counts().head(5))
    print('-' * 50)

## Step 10: Quick Visualizations to Confirm Insights

In [ ]:
#VISUAL 1: Revenue by Region
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

region_rev = df.groupby('Region')['Total Revenue'].sum().sort_values()
region_rev.plot(kind='barh', ax=axes[0], color='#FF9900')
axes[0].set_title('Total Revenue by Region', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Revenue (USD)')

channel_rev = df.groupby('Sales Channel')['Total Revenue'].sum()
channel_rev.plot(kind='pie', ax=axes[1], colors=['#FF9900', '#232F3E'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Revenue by Sales Channel', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
#VISUAL 2: Sales Trend Over Time 
yearly = df.groupby('Year')['Total Revenue'].sum()
monthly = df.groupby('Month Name')['Total Revenue'].sum().reindex(
    ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

yearly.plot(kind='line', ax=axes[0], marker='o', color='#FF9900', linewidth=2.5, markersize=10)
axes[0].set_title('Yearly Revenue Trend', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Revenue (USD)')
axes[0].grid(True, alpha=0.3)

monthly.plot(kind='bar', ax=axes[1], color='#232F3E')
axes[1].set_title('Monthly Revenue Pattern (Seasonality)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Revenue (USD)')
axes[1].set_xlabel('Month')

plt.tight_layout()
plt.show()

In [ ]:
#VISUAL 3: Top Item Types by Profit
item_perf = df.groupby('Item Type').agg({
    'Total Revenue': 'sum',
    'Total Profit': 'sum',
    'Profit Margin %': 'mean'
}).sort_values('Total Profit', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
item_perf['Total Profit'].plot(kind='barh', ax=ax, color='#FF9900')
ax.set_title('Total Profit by Item Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Profit (USD)')
plt.tight_layout()
plt.show()

print('\n📊 Item Type Performance:')
print(item_perf.round(2))

## Step 11: Export Cleaned Dataset for Power BI

In [ ]:
#EXPORT CLEAN CSV
df.to_csv('Amazon_Sales_Clean.csv', index=False)

print(f'✅ Clean dataset exported: Amazon_Sales_Clean.csv')
print(f'   Rows: {len(df):,}')
print(f'   Columns: {df.shape[1]}')
print('\n🎯 Ready to import into Power BI!')